In [ ]:
# Import packages
import pandas as pd
import numpy as np

from functions import *

In [ ]:
# Read in and check data
d = pd.read_csv("~/ml/data/kaggle_cancer/lung_cancer_mortality_data_large.csv")

print(f"Dataset Size: {d.shape[0]} Columns, {d.shape[1]} Rows\n")

print(f"Column Names:")
for i in d.columns:
    print(i)

d.head()

In [ ]:
# Define target
target = "survived"
d[target].value_counts() / len(d)

# Create Year/Month Column
d['year_month'] = d['end_treatment_date'].str[0:4] + "/" + d['end_treatment_date'].str[5:7]

In [ ]:
# Target distribution
tab = aggregate_table(
    data=d, 
    factor=target, 
    columns=['id'],
    aggs=['count']
    )

fig = plotly_plot(
    x = tab[target],
    y_list = [tab['id']],
    y_names = ['Count'],
    secondary_y = [False],
    plot_type = ['bar'],
    colours = ["#b3b5ff"],
    title = 'Volume by Survival Flag',
    x_title = 'Survived',
    y_title = 'Row Count'
    )

display(fig)

# Target over time
tab = aggregate_table(
    data=d, 
    factor='year_month', 
    columns=['id', target],
    aggs=['count', 'mean']
    )

fig = plotly_plot(
    x = tab['year_month'], 
    y_list = [tab['id'], tab[target]], 
    y_names = ['Count', 'Surivived'],
    secondary_y = [False, True], 
    plot_type = ['bar', 'scatter'], 
    colours = ['#b3b5ff', "#c3604d"],
    title = 'Survival Rate by End of Treatment Date',
    x_title = 'End of Treatment Date',
    y_title = 'Row Count',
    y2_title = 'Survival Rate',
    y_range = [0, 80000],
    y2_range = [0.18, 0.24],
    # x_range = ['2020-01-01', '2024-11-30']
    )

display(fig)

In [ ]:
# Create feature list and split into numerical and categorical types
features = [
    'gender',
    'country',
    'cancer_stage',
    'family_history',
    'smoking_status',
    'bmi',
    'cholesterol_level',
    'hypertension',
    'asthma',
    'cirrhosis',
    'other_cancer',
    'treatment_type',
]

d[features].dtypes

cats = [i for i in features if d[i].dtype == 'object']
nums = [i for i in features if i not in cats]

print("Numerical Features: ")
for i in nums:
    print(i)

print("\nCategorical Features: ")
for i in cats:
    print(i)

In [ ]:
# Summary statistics
display(numerical_summary(d, nums))
display(categorical_summary(d, cats))

In [ ]:
# Distributions by feature
for i in features:
    df = d.copy()

    if (i in nums) & (len(df[i].unique()) > 2):
        ignore, bands = pd.cut(df[i], bins=20, retbins=True)
        labels = [f"{bands[j]} to {bands[j+1]}" for j in range(len(bands)-1)]
        df[i] = pd.cut(df[i], bins=bands, labels = labels)

    tab = aggregate_table(
        data=df, 
        factor=i, 
        columns=['id'],
        aggs=['count']
        )

    fig = plotly_plot(
        x = tab[i], 
        y_list = [tab['id']], 
        y_names = ['Count'],
        secondary_y = [False], 
        plot_type = ['bar'], 
        colours = ['blue'],
        title = f'Row Count by {i}',
        x_title = i,
        y_title = 'Row Count',
        # y_range = [0, 80000],
        # y2_range = [0.18, 0.24],
        # x_range = ['2020-01-01', '2024-11-30']
        )

    display(fig)

In [ ]:
# Distributions by feature
for i in features:
    df = d.copy()
    y_list = []

    if (i in nums) & (len(df[i].unique()) > 2):
        ignore, bands = pd.qcut(df[i], q=4, retbins=True)
        labels = [f"{bands[j]} to {bands[j+1]}" for j in range(len(bands)-1)]
        df[i] = pd.cut(df[i], bins=bands, labels = labels)
    
    if (i in cats) & (len(df[i].unique()) > 9):
        vals = df[i].value_counts()
        vals = list(vals[:9].index)
        mapping = {i:i for i in vals}
        df[i] = df[i].map(mapping)
        df[i] = df[i].fillna('Other')

    tab = aggregate_table(
        data=df, 
        factor='year_month', 
        columns=['id'],
        aggs=['count']
        )
    
    tab.columns = ['year_month', 'Count']
    levels = list(df[i].unique())

    for j in levels:
        tab2 = aggregate_table(
            data=df[df[i] == j], 
            factor='year_month', 
            columns=['id'],
            aggs=['count']
            )
        
        tab2.columns = ['year_month', j]
        # tab['id'] = tab['id'] / len(df)

        tab = pd.merge(tab, tab2, on='year_month', how='left')
        tab[j] = tab[j] / tab['Count']
    
        y_list.append(tab[j])
        levels = [str(i) for i in levels]
    
    fig = make_subplots(rows=min(len(levels),4), cols=1)
    for a in range(len(y_list[:4])):
        fig.add_trace(go.Scatter(name=levels[a], x=tab['year_month'], y=y_list[a]), row=a+1, col=1)
    fig.update_layout(title_text=f"Feature Distribution over Time - {i.title()}", width=1000, height=1000)

    display(fig)

In [ ]:
y_list